# ARTGL v2.0: run all experiments

Run the cells from top to bottom. Each experiment cell can be re-run at any time: finished
runs are saved in `RESULTS/<dataset>/runs/` and skipped, so after a crash, a disconnect or a
restart you just run the same cell again and it continues where it stopped.

Works in local Jupyter (keep this notebook in the code folder) and in Google Colab (set
`USE_COLAB = True` below; code, data and results then live on your Google Drive).

## 1. Settings

In [ ]:
USE_COLAB = False      # True in Google Colab

# Folder that contains experiments.py (the unzipped Adaptive-Regularized-v2.0 folder)
CODE_DIR = "."
# Where the datasets are (see the README for which files)
DATA_DIR = "data"
# Where results are written (keep it on Drive in Colab, so it survives disconnects)
RESULTS = "results"

SEEDS = "0 1 2 3 4"

In [ ]:
import os, sys, subprocess
if USE_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = "/content/drive/MyDrive/ARTGL/Adaptive-Regularized-v2.0"
    DATA_DIR = "/content/drive/MyDrive/ARTGL/data"
    RESULTS  = "/content/drive/MyDrive/ARTGL/results"
os.chdir(CODE_DIR)
os.makedirs(RESULTS, exist_ok=True)
print("working in", os.getcwd())

## 2. Install and check the GPU

In [ ]:
# Local Jupyter: install PyTorch with CUDA first, following https://pytorch.org/get-started/locally/
# (Colab already has PyTorch with CUDA.)
!{sys.executable} -m pip install -q torch_geometric pandas scikit-learn tqdm matplotlib xgboost

In [ ]:
import torch, torch_geometric
print("PyTorch", torch.__version__, "| PyG", torch_geometric.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - training will be very slow")

## 3. Tests (must all pass before any real run)

The second test also runs the whole pipeline on small synthetic data; it takes a few minutes.
If anything fails, stop here and send me the error.

In [ ]:
!{sys.executable} tests/test_data_pipeline.py
!{sys.executable} tests/test_model_torch.py

## 4. Prepare the data from your downloaded zips

Set `DOWNLOADS` to the folder that holds the zip files (zips inside zips are fine). The script recognises the needed files by their contents, extracts only those into `DATA_DIR`, and lists what it skipped. Run it first with `--list` to preview.


In [ ]:
DOWNLOADS = r"C:\Users\you\Downloads\datasets"   # <-- change this
!{sys.executable} prepare_data.py --source "{DOWNLOADS}" --dest "{DATA_DIR}" --list


In [ ]:
!{sys.executable} prepare_data.py --source "{DOWNLOADS}" --dest "{DATA_DIR}"


### Check that the data files are found


In [ ]:
import glob
UNSW  = f"{DATA_DIR}/UNSW-NB15/UNSW-NB15_[1-4].csv"
CIC17 = f"{DATA_DIR}/CICIDS2017/TrafficLabelling/*.csv"
CIC18 = f"{DATA_DIR}/CSE-CIC-IDS2018/Processed Traffic Data for ML Algorithms/*.csv"
for name, pattern, expected in [("UNSW-NB15", UNSW, 4), ("CICIDS2017", CIC17, 8), ("CSE-CIC-IDS2018", CIC18, 10)]:
    n = len(glob.glob(pattern))
    print(f"{name:16s} {n:2d} files (expected {expected})", "" if n == expected else "  <-- check")

In [ ]:
def run(config, suites, data):
    cmd = (f'{sys.executable} experiments.py --config configs/{config}.json --suites {suites} '
           f'--seeds {SEEDS} --output_dir "{RESULTS}" --data_paths "{data}"')
    print(cmd)
    subprocess.run(cmd, shell=True, check=True)

## 5. UNSW-NB15

Run one cell at a time, in this order. The first cell also loads the data and builds the
graphs (cached for later cells). Afterwards, open `RESULTS/unsw_nb15/data_summary.json`
to check the record counts and class balance in each split.

In [ ]:
run("unsw_nb15", "main", UNSW)          # Table 3 + Table 11

In [ ]:
run("unsw_nb15", "ablation noise", UNSW) # Tables 4, 5, 6

In [ ]:
run("unsw_nb15", "adversarial", UNSW)    # Table 7

In [ ]:
run("unsw_nb15", "sensitivity", UNSW)    # Table 8 (100 trainings; the longest step)

## 6. CICIDS2017

In [ ]:
run("cicids2017", "main", CIC17)          # Table 3

## 7. Cross-dataset (Table 9)

In [ ]:
def cross(train, test):
    args = " ".join(f"configs/{t}.json" for t in train)
    cmd = (f'{sys.executable} cross_dataset.py --train {args} --test configs/{test}.json '
           f'--seeds {SEEDS} --output_dir "{RESULTS}/cross_dataset"')
    print(cmd)
    subprocess.run(cmd, shell=True, check=True)

# cross_dataset.py reads data paths from the config files, so point them at DATA_DIR once:
import json
for name, pattern in [("unsw_nb15", UNSW), ("cicids2017", CIC17), ("cse_cicids2018", CIC18)]:
    path = f"configs/{name}.json"
    cfg = json.load(open(path))
    cfg["data_paths"] = [pattern]
    json.dump(cfg, open(path, "w"), indent=2)

cross(["unsw_nb15"], "cicids2017")
cross(["cicids2017"], "unsw_nb15")
cross(["unsw_nb15", "cicids2017"], "cse_cicids2018")

## 8. Tables and figures

In [ ]:
!{sys.executable} aggregate.py --results "{RESULTS}/unsw_nb15" --cross "{RESULTS}/cross_dataset"
!{sys.executable} aggregate.py --results "{RESULTS}/cicids2017"
!{sys.executable} plots.py --tables "{RESULTS}/unsw_nb15/tables" --dataset UNSW-NB15
!{sys.executable} plots.py --tables "{RESULTS}/cicids2017/tables" --dataset CICIDS2017

In [ ]:
import pandas as pd
for f in sorted(glob.glob(f"{RESULTS}/unsw_nb15/tables/*.csv")):
    print("\n==", os.path.basename(f))
    display(pd.read_csv(f).round(4))

## 9. Send back

Zip and send me `RESULTS/unsw_nb15/tables`, `RESULTS/cicids2017/tables`, every `*_per_seed.csv`,
and both `data_summary.json` files. I will put the numbers into the manuscript and draft the
response letter. (Do not send the `runs/` and `cache/` folders; they are large.)

In [ ]:
import shutil, tempfile
pack = os.path.join(RESULTS, "send_to_claude")
os.makedirs(pack, exist_ok=True)
for ds in ["unsw_nb15", "cicids2017"]:
    for f in glob.glob(f"{RESULTS}/{ds}/*.csv") + glob.glob(f"{RESULTS}/{ds}/*.json") + glob.glob(f"{RESULTS}/{ds}/tables/*"):
        dest = os.path.join(pack, ds, os.path.relpath(f, f"{RESULTS}/{ds}"))
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(f, dest)
for f in glob.glob(f"{RESULTS}/cross_dataset/*/*"):
    if f.endswith((".csv", ".json")):
        dest = os.path.join(pack, "cross_dataset", os.path.relpath(f, f"{RESULTS}/cross_dataset"))
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(f, dest)
print(shutil.make_archive(pack, "zip", pack))